# Fractal-Driven Predictive System — CSV Pipeline
**Feed any CSV → get predictions, IFS attractor, and forward projections.**

## Quick start
1. Run **Cell 1** (setup)
2. Run **Cell 2** to check your CSV
3. Run **Cell 3** to train and predict
4. Optionally run **Cell 4** with your own CSV

---

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ════════════════════════════════════════════════════════════════
# ----- For Google Colab only: upload your project zip first -----
# from google.colab import files
# uploaded = files.upload()   # upload fractal_predictive_system.zip
# import zipfile
# with zipfile.ZipFile('fractal_predictive_system.zip') as z:
#     z.extractall('.')
# ----------------------------------------------------------------

import sys, os
from pathlib import Path

# Point to project root — adjust if your folder is elsewhere
PROJECT = Path('fractal_predictive_system')   # <-- change if needed
sys.path.insert(0, str(PROJECT / 'modules'))

# Make matplotlib show plots inline
import matplotlib
matplotlib.use('inline')       # Jupyter
# matplotlib.use('inline')     # same for Colab

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from neural_net import NeuralNetwork
from ifs_engine  import IFSPredictor

print('✓ Setup complete')
print(f'  NumPy   : {np.__version__}')
print(f'  Pandas  : {pd.__version__}')
print(f'  Project : {PROJECT.resolve()}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — PREVIEW YOUR CSV
# ════════════════════════════════════════════════════════════════
# Change the path below to whichever CSV you want to use

CSV_FILE = PROJECT / 'data' / 'birth_rate.csv'     # <-- change me

df = pd.read_csv(CSV_FILE)
print(f'File   : {CSV_FILE.name}')
print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
df.head(10)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — RUN THE FULL PIPELINE
# ════════════════════════════════════════════════════════════════

# Import the pipeline (reads csv_pipeline.py from project root)
sys.path.insert(0, str(PROJECT))

# Override matplotlib backend before importing pipeline
import matplotlib
matplotlib.use('Agg')    # saves to file
# matplotlib.use('inline') # shows inline — use this in Colab/Jupyter

from csv_pipeline import run_csv_pipeline

# ── Choose a dataset ─────────────────────────────────────────────

# OPTION A — Birth rate
results = run_csv_pipeline(
    csv_path   = str(PROJECT / 'data' / 'birth_rate.csv'),
    value_col  = 'world_avg',       # column to predict
    year_col   = 'year',
    epochs     = 600,
    n_future   = 5,
    color      = '#1D9E75',
    save_fig   = True,
    save_proj  = True,
)

# OPTION B — Climate anomaly (uncomment to use)
# results = run_csv_pipeline(
#     csv_path  = str(PROJECT / 'data' / 'climate_anomaly.csv'),
#     value_col = 'global_anomaly_c',
#     epochs    = 600,
#     color     = '#D85A30',
# )

# OPTION C — Conflict data (uncomment to use)
# results = run_csv_pipeline(
#     csv_path  = str(PROJECT / 'data' / 'conflict_data.csv'),
#     value_col = 'active_conflicts',
#     epochs    = 800,
#     color     = '#C0392B',
# )

print('\nProjection results:')
proj = results['projection']
for y, v, lo, hi in zip(proj['years'], proj['values'],
                          proj['lower'], proj['upper']):
    print(f'  {y}: {v:.3f}  [{lo:.3f} – {hi:.3f}]')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — YOUR OWN CSV
# ════════════════════════════════════════════════════════════════
# --- Colab: upload your file first ---
# from google.colab import files
# uploaded = files.upload()   # pick your CSV
# MY_CSV = list(uploaded.keys())[0]

# --- Jupyter: just set the path ---
MY_CSV     = str(PROJECT / 'data' / 'template_custom.csv')  # <-- your file
MY_COL     = 'value'     # <-- column you want to predict
MY_EPOCHS  = 600
MY_FUTURE  = 5

my_results = run_csv_pipeline(
    csv_path  = MY_CSV,
    value_col = MY_COL,
    epochs    = MY_EPOCHS,
    n_future  = MY_FUTURE,
)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — VIEW THE SAVED FIGURE (Jupyter)
# ════════════════════════════════════════════════════════════════
from IPython.display import Image, display
import glob

# Show the most recently saved figure
figs = sorted(glob.glob(str(PROJECT / 'outputs' / 'csv_pipeline_*.png')))
if figs:
    display(Image(figs[-1], width=900))
    print(f'Showing: {figs[-1]}')
else:
    print('No figures saved yet — run Cell 3 first.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — LOAD AND INSPECT PROJECTION CSV
# ════════════════════════════════════════════════════════════════
import glob

proj_files = sorted(glob.glob(str(PROJECT / 'outputs' / 'projection_*.csv')))
if proj_files:
    df_proj = pd.read_csv(proj_files[-1])
    print(f'Projection file: {proj_files[-1]}')
    display(df_proj)
else:
    print('No projection CSV yet — run Cell 3 first.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — RUN MULTIPLE COLUMNS FROM THE SAME CSV
# ════════════════════════════════════════════════════════════════
# Useful for conflict_data.csv which has 4 columns to predict

MULTI_CSV  = str(PROJECT / 'data' / 'conflict_data.csv')
COLUMNS    = ['active_conflicts', 'battle_deaths_thousands',
              'high_intensity_wars', 'conflict_countries']
COLORS     = ['#C0392B', '#E67E22', '#2980B9', '#1D9E75']

all_results = {}
for col, color in zip(COLUMNS, COLORS):
    print(f'\n>>> Column: {col}')
    r = run_csv_pipeline(
        csv_path  = MULTI_CSV,
        value_col = col,
        epochs    = 600,
        n_future  = 5,
        color     = color,
    )
    all_results[col] = r

print('\n=== All projections ===')
for col, r in all_results.items():
    vals = r['projection']['values']
    yrs  = r['projection']['years']
    print(f'\n{col}:')
    for y, v in zip(yrs, vals):
        print(f'  {y}: {v:.2f}')

---
## CSV Format Reference

Your CSV must have at minimum two columns: **year** and **one numeric column**.

```
year,value
2000,21.4
2001,21.0
...
```

You can have multiple value columns — just change `value_col` when calling the pipeline.

| Column name | Required? | Notes |
|---|---|---|
| `year` | Yes (or specify `year_col`) | Integer years |
| Any numeric column | Yes (at least one) | The signal to predict |
| Extra columns | Optional | Ignored unless specified |

### Tips
- Minimum **6 data points**, ideally 20+
- Values can be any scale — the model normalises internally
- NaN rows are dropped automatically
- Time gaps are allowed (the model uses row order, not year arithmetic)
